# 4.1 · Herramientas con `@tool` y servidores MCP in-process

**Problema:** en el bloque 3 portamos las herramientas deprisa. Ahora queremos hacerlo bien: esquemas claros, errores útiles y metadatos.

**Al terminar tendrás:** un servidor `soporte` con tres herramientas bien definidas.

Una herramienta mal definida es la causa número uno de agentes que hacen cosas raras.

## Preparación

`imprimir()` ya la conocemos del 3.1, así que la hemos sacado del notebook a `taller_utils.py`, en la raíz. A partir de aquí, lo que ya se ha enseñado se mueve a ese módulo y el siguiente notebook lo importa.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import herramientas
from taller_utils import imprimir

from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, ResultMessage, ToolAnnotations,
)

herramientas.restaurar_datos()

## Anatomía de `@tool`

```python
@tool(nombre, descripción, esquema, annotations=...)
async def f(args: dict) -> dict:
    return {"content": [{"type": "text", "text": "..."}]}       # y "is_error": True si falla
```

El esquema admite tres formas:

| Forma | Ejemplo | Cuándo |
|---|---|---|
| Mapa de tipos | `{"cliente": str}` | prototipos. Todos los campos son obligatorios. |
| `TypedDict` | `class Args(TypedDict): cliente: Annotated[str, "Nombre del cliente"]` | si ya trabajas con tipos de Python |
| JSON Schema completo | `{"type": "object", "properties": {...}, "required": [...]}` | cuando hacen falta **enums**, **campos opcionales** y **descripciones por campo** |

Para un agente de verdad, casi siempre JSON Schema: cada descripción de campo es una pista más para el modelo.

Dos funciones auxiliares para no repetir el formato de la respuesta:

In [ ]:
def texto(dato):
    return {"content": [{"type": "text", "text": json.dumps(dato, ensure_ascii=False)}]}

def error(mensaje):
    return {"content": [{"type": "text", "text": mensaje}], "is_error": True}

`buscar_incidencias`, ahora con JSON Schema: los tres filtros son **opcionales** (no hay `required`), el estado es un `enum` y cada campo tiene su descripción. La descripción de la herramienta dice cuándo usarla, qué devuelve y qué pasa si no hay resultados.

In [ ]:
@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes. Úsala siempre antes de crear una incidencia y cuando "
      "pregunten por incidencias de un cliente o por su estado. Todos los filtros son opcionales y se "
      "combinan. Devuelve una lista JSON con id, cliente, asunto, estado, prioridad y fecha; [] si no "
      "hay coincidencias.",
      {
          "type": "object",
          "properties": {
              "texto": {"type": "string", "description": "Palabras clave a buscar en asunto y descripción"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, p. ej. 'ACME'"},
          },
      })
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

`crear_incidencia`: aquí todos los campos son obligatorios y la descripción deja claro **cuándo no** usarla.

In [ ]:
@tool("crear_incidencia",
      "Crea una incidencia nueva. Úsala SOLO después de comprobar con buscar_incidencias que no existe "
      "ya una abierta sobre el mismo problema para el mismo cliente.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string"},
              "asunto": {"type": "string", "description": "Resumen en menos de 10 palabras"},
              "descripcion": {"type": "string"},
              "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
          },
          "required": ["cliente", "asunto", "descripcion", "prioridad"],
      })
async def crear_incidencia(args):
    return texto(herramientas.crear_incidencia(**args))

Cada herramienta se puede probar sola, sin agente:

In [ ]:
await buscar_incidencias.handler({"cliente": "ACME", "estado": "abierta"})

## Errores que ayudan al modelo

Un error no es solo "algo ha ido mal": es un mensaje para el modelo, que lo leerá y decidirá qué hacer. Por eso debe decir **qué hacer a continuación**.

In [ ]:
@tool("cerrar_incidencia", "Cierra una incidencia abierta por su id (formato INC-123).",
      {"id": str})
async def cerrar_incidencia(args):
    inc = herramientas.obtener(args["id"])
    if inc is None:
        return error(f"No existe la incidencia {args['id']}. "
                     "Usa buscar_incidencias para obtener ids válidos.")
    if inc["estado"] == "cerrada":
        return error(f"La incidencia {args['id']} ya estaba cerrada. No hay que hacer nada más.")
    return texto(herramientas.cerrar_incidencia(args["id"]))

In [ ]:
await cerrar_incidencia.handler({"id": "INC-999"})

Compáralo con un error típico, `KeyError: 'INC-999'`: es verdad, pero no le dice al modelo cómo salir del problema. Lo veremos funcionar con el agente al final del notebook.

## Anotaciones

Las anotaciones son metadatos sobre el comportamiento de la herramienta, que se envían junto a su descripción:

| Anotación | Significa |
|---|---|
| `readOnlyHint=True` | solo lee, no cambia nada |
| `destructiveHint=True` | puede hacer cambios difíciles de deshacer |
| `idempotentHint=True` | llamarla dos veces con lo mismo tiene el mismo efecto que una |

Sirven al propio agente (sabe qué herramientas son seguras) y a las políticas de permisos. **No imponen nada**: son pistas. Lo que se impone, se impone con permisos y hooks (4.4).

Redefinimos `buscar_incidencias` y `cerrar_incidencia` con su anotación:

In [ ]:
@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes. Úsala siempre antes de crear una incidencia y cuando "
      "pregunten por incidencias de un cliente o por su estado. Todos los filtros son opcionales y se "
      "combinan. Devuelve una lista JSON con id, cliente, asunto, estado, prioridad y fecha; [] si no "
      "hay coincidencias.",
      {
          "type": "object",
          "properties": {
              "texto": {"type": "string", "description": "Palabras clave a buscar en asunto y descripción"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, p. ej. 'ACME'"},
          },
      },
      annotations=ToolAnnotations(readOnlyHint=True))
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

In [ ]:
@tool("cerrar_incidencia", "Cierra una incidencia abierta por su id (formato INC-123).",
      {"id": str},
      annotations=ToolAnnotations(destructiveHint=True))
async def cerrar_incidencia(args):
    inc = herramientas.obtener(args["id"])
    if inc is None:
        return error(f"No existe la incidencia {args['id']}. "
                     "Usa buscar_incidencias para obtener ids válidos.")
    if inc["estado"] == "cerrada":
        return error(f"La incidencia {args['id']} ya estaba cerrada. No hay que hacer nada más.")
    return texto(herramientas.cerrar_incidencia(args["id"]))

## El servidor in-process

*In-process* significa que el servidor corre **dentro de nuestro programa Python**: sin procesos aparte ni red. Es lo más simple y lo más rápido para herramientas propias.

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia, cerrar_incidencia])

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt="Eres un agente de soporte de nivel 1. Respondes en español y de forma breve.",
    mcp_servers={"soporte": soporte},
    tools=[],
    allowed_tools=[
        "mcp__soporte__buscar_incidencias",
        "mcp__soporte__crear_incidencia",
        "mcp__soporte__cerrar_incidencia",
    ],
    max_turns=10,
    setting_sources=[],
)

Recuerda el nombre: `mcp__<servidor>__<herramienta>`. Es el que va en `allowed_tools`.

Ahora sí, el error útil en acción. Le pedimos cerrar una incidencia con un id que no existe:

In [ ]:
async for msg in query(prompt="El cliente ACME confirma que ya está resuelto lo del logo descolocado en el "
                              "informe mensual: es la INC-999. Ciérrala.", options=options):
    imprimir(msg)

El modelo recibe el error, busca y se corrige solo: encuentra el id correcto y la cierra.

> A veces el modelo desconfía del id y busca **antes** de intentar cerrar, así que ni siquiera llega a ver el error. También es buena conducta; reejecuta la celda para ver el otro camino.

In [ ]:
herramientas.restaurar_datos()

## Resumen

- `@tool(nombre, descripción, esquema)` sobre una función `async`. Para agentes reales, esquema en JSON Schema: enums, opcionales y descripciones por campo.
- Los errores se devuelven con `is_error: True` y dicen **qué hacer a continuación**.
- Las anotaciones (`readOnlyHint`, `destructiveHint`) son metadatos, no permisos.
- Un servidor in-process agrupa las herramientas y corre dentro del propio programa.

**Para el siguiente vídeo:** esto son herramientas nuestras. ¿Y si alguien ya ha hecho la integración que necesito?

## 🔸 Reto

Añade una herramienta `buscar_kb(consulta)` que busque en `datos/kb/` y devuelva el título y el fragmento más relevante de cada artículo. Dásela al agente y pregúntale qué le dirías a un cliente que no puede exportar a PDF.

La búsqueda ya está escrita: `herramientas.buscar_kb(consulta)`.